# i need to change my design like cpu -> gpu structure, not gpu dedicated design

In [ ]:
import torch
import torch.nn as nn
import optuna
import pandas as pd
from optuna import Trial
import torchmetrics

torch.manual_seed(42)
device = "cuda"

## 13

In [2]:
x = torch.tensor(1.2, requires_grad=True)
y = torch.tensor(3.4, requires_grad=True)

f = torch.sin((x**2)*y)
f.backward()

print("Gradient of x:", x.grad)
print("Gradient of y:", y.grad)


Gradient of x: tensor(1.4899)
Gradient of y: tensor(0.2629)


## 14

In [ ]:
class InhuDense(nn.Module):
    def __init__(self, input_num:int, output_num:int):
        super().__init__()
        self.input_w = nn.Parameter(torch.randn(input_num, output_num, device=device))
        self.input_b = nn.Parameter(torch.randn(output_num, device=device))
    
    def forward(self, x):
        return torch.relu(x @ self.input_w + self.input_b)

## 15 classification

In [4]:
from sklearn.datasets import fetch_covtype
from torch.utils.data import Dataset

# Load the Covertype dataset
covtype = fetch_covtype()
X = covtype['data']
y = covtype['target']

# Create a custom PyTorch Dataset
class CovtypeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).to(device)
        self.y = torch.tensor(y - 1, dtype=torch.long).to(device)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

covtype_dataset = CovtypeDataset(X, y)

covtype_df = pd.DataFrame(X, columns=covtype['feature_names'])
covtype_df

,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type_30,Soil_Type_31,Soil_Type_32,Soil_Type_33,Soil_Type_34,Soil_Type_35,Soil_Type_36,Soil_Type_37,Soil_Type_38,Soil_Type_39
0,2596.0,51.0,3.0,258.0,0.0,510.0,221.0,232.0,148.0,6279.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2590.0,56.0,2.0,212.0,-6.0,390.0,220.0,235.0,151.0,6225.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2804.0,139.0,9.0,268.0,65.0,3180.0,234.0,238.0,135.0,6121.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2785.0,155.0,18.0,242.0,118.0,3090.0,238.0,238.0,122.0,6211.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2595.0,45.0,2.0,153.0,-1.0,391.0,220.0,234.0,150.0,6172.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581007,2396.0,153.0,20.0,85.0,17.0,108.0,240.0,237.0,118.0,837.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581008,2391.0,152.0,19.0,67.0,12.0,95.0,240.0,237.0,119.0,845.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581009,2386.0,159.0,17.0,60.0,7.0,90.0,236.0,241.0,130.0,854.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581010,2384.0,170.0,15.0,60.0,5.0,90.0,230.0,245.0,143.0,864.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
np.unique_counts(y)

UniqueCountsResult(values=array([1, 2, 3, 4, 5, 6, 7], dtype=int32), counts=array([211840, 283301,  35754,   2747,   9493,  17367,  20510]))

In [5]:
batch_size = 128
num_worker = 0
persistent_worker = False
pin_memory = False

n_epoch = 5
lr = 0.001

In [6]:
from torch.utils.data import DataLoader
from torch.utils.data import random_split

data_train, data_val, data_test  = random_split(covtype_dataset, [0.7, 0.2, 0.1])

train_loader = DataLoader(data_train, shuffle=True, pin_memory=pin_memory, num_workers=num_worker, persistent_workers=persistent_worker, batch_size=batch_size)
val_loader = DataLoader(data_val, pin_memory=pin_memory, num_workers=num_worker, persistent_workers=persistent_worker, batch_size=batch_size)
test_loader = DataLoader(data_test, pin_memory=pin_memory, num_workers=num_worker, persistent_workers=persistent_worker, batch_size=batch_size)

In [ ]:
from torch.optim import Optimizer

class Classification(nn.Module):
    def __init__(self, n_inputs:int = 54, n_hidden_num_list:list[int] = [], n_classes = 7) -> None:
        super().__init__()
        hidden_layers:list[nn.Module] = []
        for idx in range(1,len(n_hidden_num_list)):
            new_layer = nn.Linear(n_hidden_num_list[idx-1], n_hidden_num_list[idx])
            hidden_layers.append(new_layer)
            hidden_layers.append(nn.ReLU())

        self.mlp = nn.Sequential(
            nn.Linear(n_inputs, n_hidden_num_list[0]), nn.ReLU(),
            *hidden_layers,
            nn.Linear(n_hidden_num_list[-1], n_classes)
        )
    
    def forward(self, X):
        return self.mlp(X)


def train(model:nn.Module, criterion, optimizer:Optimizer, dataloader:DataLoader, n_epoch:int = 1):
    model.to(device)
    model.train()
    for i in range(n_epoch):
        total_loss:float = 0.0
        for X,y in dataloader:
            optimizer.zero_grad()
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            y_pred = model(X)
            loss = criterion(y_pred, y)
            loss.backward()
            total_loss += loss.item()
            optimizer.step()
            
        print(f"epoch:{i}, mean_loss:{total_loss/len(dataloader)}")
    
def evaluate(model:nn.Module, metric_fn, dataloader:DataLoader, aggre_fn = torch.mean):
    model.to(device)
    model.eval()
    metrics = []
    for X,y in dataloader:
        X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.no_grad():
            y_pred = model(X)
            metrics.append(metric_fn(y_pred, y))
    
    metric_fn.reset()
    return aggre_fn(torch.stack(metrics))

In [ ]:
def objective(trial:Trial, criterion, dataloader:DataLoader):
    accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    layer_nums = trial.suggest_int("layer_nums", 1,4)
    n_hidden_num_list = [trial.suggest_int(f"n_hidden_{i}", 32, 512) for i in range(layer_nums)]
    
    model = Classification(n_hidden_num_list=n_hidden_num_list).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr = lr)

    for i in range(n_epoch):
        train(model, criterion, optimizer, dataloader, lr)
        val_acc = evaluate(model, accuracy, val_loader)
        trial.report(val_acc.item(), step=i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
    
    return val_acc.item()
        

In [ ]:
from backend.engine import engine, db_url

criterion = nn.CrossEntropyLoss()
sampler = optuna.samplers.TPESampler(seed=42)
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, interval_steps=1)

storage = optuna.storages.RDBStorage(url = db_url)
study = optuna.create_study(
    study_name = "covtype_study",
    storage=storage,
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    load_if_exists=False
)

[I 2026-05-24 10:30:59,124] Using an existing study with name 'covtype_study' instead of creating a new one.


In [10]:
study.optimize(lambda trial: objective(trial, criterion, train_loader), n_trials=20, n_jobs=1)

print("Best trial:")
print(f"  Accuracy: {study.best_value:.4f}")
print(f"  Params:   {study.best_params}")

epoch:1, mean_loss:0.8040021744968757
epoch:1, mean_loss:0.6279726003685262
epoch:1, mean_loss:0.5684986507521251
epoch:1, mean_loss:0.521213041096832
epoch:1, mean_loss:0.48633817100839993


[I 2026-05-24 10:32:04,258] Trial 14 finished with value: 0.808977484703064 and parameters: {'lr': 0.00031489116479568613, 'layer_nums': 4, 'n_hidden_0': 384, 'n_hidden_1': 319, 'n_hidden_2': 107, 'n_hidden_3': 107}. Best is trial 14 with value: 0.808977484703064.


epoch:1, mean_loss:1.0030966264560883
epoch:1, mean_loss:0.7519866154019829
epoch:1, mean_loss:0.6867034221648269
epoch:1, mean_loss:0.649620208718103
epoch:1, mean_loss:0.6262099321361906


[I 2026-05-24 10:33:05,350] Trial 15 finished with value: 0.7375701665878296 and parameters: {'lr': 1.7073967431528103e-05, 'layer_nums': 4, 'n_hidden_0': 321, 'n_hidden_1': 372, 'n_hidden_2': 41, 'n_hidden_3': 498}. Best is trial 14 with value: 0.808977484703064.


epoch:1, mean_loss:2.256015179834702
epoch:1, mean_loss:1.2074532514476717
epoch:1, mean_loss:1.2073738187307679
epoch:1, mean_loss:1.2074647714800621
epoch:1, mean_loss:1.2073549770048537


...

epoch:1, mean_loss:0.7946726972479877
epoch:1, mean_loss:0.6056043014839957
epoch:1, mean_loss:0.5372144085870291


[I 2026-05-24 10:41:14,858] Trial 32 pruned. 


epoch:1, mean_loss:1.0962461228075082


[I 2026-05-24 10:41:25,658] Trial 33 pruned. 


Best trial:
  Accuracy: 0.8196
  Params:   {'lr': 0.0006672367170464204, 'layer_nums': 4, 'n_hidden_0': 128, 'n_hidden_1': 279, 'n_hidden_2': 316, 'n_hidden_3': 54}


In [ ]:
best_params = study.best_params
best_lr = best_params["lr"]
best_hidden = [best_params[f"n_hidden_{i}"] for i in range(best_params["layer_nums"])]

best_model = Classification(n_hidden_num_list=best_hidden)
optimizer = torch.optim.AdamW(best_model.parameters(), lr=best_lr)
train(best_model, criterion, optimizer, train_loader, n_epoch)

torch.save({
    "model_state_dict": best_model.state_dict(),
    "hidden_layers": best_hidden,
    "lr": best_lr,
}, "best_model.pth")
print("Model saved to best_model.pth")

epoch:5, mean_loss:0.771758951692179
epoch:5, mean_loss:0.5871512681770955
epoch:5, mean_loss:0.5158069176895953
epoch:5, mean_loss:0.4686006451737138
epoch:5, mean_loss:0.43012816075861565
Model saved to best_model.pth


In [ ]:
metric_fn = torchmetrics.Accuracy(task="multiclass", num_classes=7).to(device)
evaluate(best_model, metric_fn, test_loader)

tensor(0.8150, device='cuda:0')